# Fifth PyCUDA example: a CUDA kernel with GPUArray

This final notebook in the five-example sequence combines **the exact CUDA kernel from the first example** with the `GPUArray` storage used in the third and fourth examples.

We keep the same input values, $N=100003$, `float32` precision and 256 threads per block. `SourceModule` compiles the explicit CUDA source; `gpuarray.to_gpu`, `gpuarray.empty_like` and `get` simplify allocation and transfers.

Run the cells from top to bottom in a GPU runtime. Results and timings appear inline; no result files are exported.


## 1. Prepare the GPU runtime

In Google Colab, select **Runtime → Change runtime type → GPU** and connect. The following commands inspect the assigned GPU and CUDA compiler on the remote machine. `SourceModule` needs `nvcc`; a GPU runtime alone does not establish that the compiler is available.


In [1]:
!nvidia-smi
!nvcc --version


Thu Oct 15 05:29:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Install PyCUDA if necessary. The installation may build the package and take several minutes. Continue only after installation succeeds.


In [2]:
!pip install pycuda


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 10.8 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5281394 sha256=12f6f939f23b8ae3308db5768ac5696fd2ac8f976c3f429cf84993e13c2f9ce8
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


## 2. Initialize PyCUDA

Use the same context initialization and environment printout as in the preceding examples. Here both `SourceModule` and `gpuarray` are needed: the former compiles our CUDA source, while the latter manages the device arrays.


In [3]:
import sys
import numpy as np
import pycuda
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule
import pycuda.gpuarray as gpuarray

device = cuda.Context.get_device()
print('GPU:', device.name())
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('PyCUDA:', pycuda.VERSION_TEXT)


GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
PyCUDA: 2026.1


## 3. Choose the vector length and launch size

Use $N=100003$ and 256 threads per block, as in Chapter 2. The length is deliberately not divisible by the block size. `iDivUp(N, BLOCKSIZE)` computes the number of blocks by rounding the division upward.

The kernel uses 32-bit integer indexing. The checks below restrict the configuration to valid device limits and keep the padded launch indices within that range.


In [4]:
N = 100003
BLOCKSIZE = 256

def iDivUp(a, b):
    return (a + b - 1) // b

assert 1 <= BLOCKSIZE <= min(
    device.get_attribute(cuda.device_attribute.MAX_THREADS_PER_BLOCK),
    device.get_attribute(cuda.device_attribute.MAX_BLOCK_DIM_X),
)
assert 1 <= N <= np.iinfo(np.int32).max - (BLOCKSIZE - 1)

number_of_blocks = iDivUp(N, BLOCKSIZE)
assert number_of_blocks <= device.get_attribute(
    cuda.device_attribute.MAX_GRID_DIM_X
)
block = (BLOCKSIZE, 1, 1)
grid = (number_of_blocks, 1, 1)

print(f'Elements: {N}')
print(f'Blocks: {number_of_blocks}; threads per block: {BLOCKSIZE}')
print(f'Threads with no output element: {number_of_blocks * BLOCKSIZE - N}')


Elements: 100003
Blocks: 391; threads per block: 256
Threads with no output element: 93


## 4. Prepare the host arrays

The inputs reproduce the Chapter 2 initialization: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both are contiguous, one-dimensional `float32` arrays, matching CUDA's `float` pointers. `h_c` will receive the GPU result; `np.empty_like` allocates its storage without initializing the entries.


In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
h_c = np.empty_like(h_a)

assert h_a.dtype == h_b.dtype == h_c.dtype == np.float32
assert all(x.flags.c_contiguous for x in (h_a, h_b, h_c))
print(f'Bytes per vector: {h_a.nbytes}')


Bytes per vector: 400012


## 5. Transfer the inputs and allocate the output

`gpuarray.to_gpu` creates device arrays and copies the NumPy inputs into them. Each `GPUArray` object retains its shape, dtype and device storage. Create a separate output with `gpuarray.empty_like` and reuse it at every launch.

The uploaded original used `zeros_like`. Zero initialization is unnecessary here because the kernel writes every valid output element before it is read. `empty_like` allocates storage without filling it.

The arrays must remain alive while GPU work uses them. This example keeps all three references until the final cleanup cell.


In [6]:
d_a = gpuarray.to_gpu(h_a)
d_b = gpuarray.to_gpu(h_b)
d_c = gpuarray.empty_like(d_a)

assert d_a.shape == d_b.shape == d_c.shape == (N,)
assert d_a.dtype == d_b.dtype == d_c.dtype == np.float32
print('Device array class:', type(d_a).__name__)
print('Shape:', d_a.shape)
print('Dtype:', d_a.dtype)
print('Bytes per device array:', d_a.nbytes)


Device array class: GPUArray
Shape: (100003,)
Dtype: float32
Bytes per device array: 400012


## 6. Compile the unchanged CUDA kernel

The source below is identical to the first PyCUDA notebook and to the Chapter 2 kernel. Its arguments remain `(a, b, c, n)`; neither thread indexing nor the bounds check changes when we use `GPUArray` on the host.

`no_extern_c=True` prevents `SourceModule` from adding another linkage wrapper because this source already declares `extern "C"`.


In [7]:
kernel_source = r"""
extern "C" __global__
void vector_add(const float *a, const float *b, float *c, int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}
"""

module = SourceModule(
    kernel_source, options=["-std=c++14"], no_extern_c=True
)
vector_add = module.get_function("vector_add")


## 7. Launch the kernel using GPUArray arguments

The positional arguments are now `GPUArray` objects instead of the bare allocations used in the first example. PyCUDA obtains their existing device addresses and passes those addresses to the CUDA function. The kernel sees `float` pointers, not Python objects. Passing these arrays does not copy their contents or allocate another output.

Their underlying device allocations are also accessible through `.gpudata`, but explicit access is unnecessary for this call. Unlike `ElementwiseKernel`, this explicit kernel still requires the typed length `np.int32(N)` and our chosen block/grid dimensions. It does not automatically adapt to array strides: the contiguous, one-dimensional `float32` arrays used here match its indexing.

The explicit synchronization below follows the first example and waits for GPU completion before retrieval.


In [8]:
vector_add(d_a, d_b, d_c, np.int32(N), block=block, grid=grid)
cuda.Context.synchronize()


## 8. Retrieve and validate every output

`d_c.get(ary=h_c)` copies the result into the existing NumPy output array. Use the same finite-value check, tolerances and CPU reference as in the other notebooks. All elements are checked, including those in the partial final block.


In [9]:
d_c.get(ary=h_c)
h_reference = h_a + h_b

if not np.all(np.isfinite(h_c)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f'Validation passed for all {N} elements.')
print(f'Maximum absolute error: {max_abs_error:.3e}')


Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 9. Display the computed sum

The `c[i] (GPU)` column comes from `h_c`, the result copied from device memory. Display the first 12 and last 3 entries so that the output includes the final, partially filled block. The NumPy column is shown separately for comparison.


In [10]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')


       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 10. Optional: measure the device interval

After three warm-up launches and synchronization, CUDA events bracket 100 launches of `vector_add`. The three GPU arrays are already allocated and are reused throughout. Report the mean device interval per launch, using the same method as in the first and fourth notebooks.

Allocation, compilation, transfers, validation and printing are outside the interval. For a short kernel, device idle gaps between Python submissions may contribute. This is not an end-to-end application time or a CPU speedup measurement.

The scope differs from the second notebook's complete calls including transfers and from the third notebook's host-timed expressions including output allocation. The five printed timings must therefore not be used directly as a ranking of the interfaces.


In [11]:
REPEATS = 100
assert REPEATS > 0

for _ in range(3):
    vector_add(d_a, d_b, d_c, np.int32(N), block=block, grid=grid)
cuda.Context.synchronize()

start = cuda.Event()
stop = cuda.Event()
start.record()
for _ in range(REPEATS):
    vector_add(d_a, d_b, d_c, np.int32(N), block=block, grid=grid)
stop.record()
stop.synchronize()

elapsed_ms = start.time_till(stop)
mean_ms = elapsed_ms / REPEATS

# Retrieve and validate outside the event interval.
d_c.get(ary=h_c)
if not np.all(np.isfinite(h_c)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
print(f'Total device interval for {REPEATS} launches: {elapsed_ms:.6f} ms')
print(f'Mean device interval per launch: {mean_ms:.6f} ms')
print('Allocation, compilation, transfers and validation are excluded.')
print('Final result validated.')


Total device interval for 100 launches: 1.467136 ms
Mean device interval per launch: 0.014671 ms
Allocation, compilation, transfers and validation are excluded.
Final result validated.


## 11. Release the GPU arrays

Once GPU work is complete, remove the notebook's references to the three arrays. Their objects manage the device allocations. Do not separately free `.gpudata` while continuing to use the owning arrays. To rerun the computation after cleanup, start again from the input-transfer and output-allocation cell.


In [12]:
cuda.Context.synchronize()
del d_a, d_b, d_c
print('GPU array references released.')


GPU array references released.


## What does this fifth example establish?

Explicit CUDA kernels and `GPUArray` are compatible parts of PyCUDA. We can choose the kernel implementation and launch dimensions while using array objects to manage storage and transfers. The exact same device arrays can also be used by other GPUArray operations without a host round trip, provided their dependencies are correctly ordered.

This combination does not by itself change the numerical operation or guarantee faster execution. It changes how the Python program manages the data supplied to the kernel.

**Execution status:** supplied with cleared outputs. Notebook structure, Python syntax, input consistency and exact reuse of the first kernel have been checked. CUDA compilation and execution have not been performed in the preparation environment; run the notebook on your GPU for validation results and timings.

References: [PyCUDA GPUArray documentation](https://documen.tician.de/pycuda/array.html) and [PyCUDA kernel argument handling](https://github.com/inducer/pycuda/blob/main/pycuda/driver.py).
